# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'brand website',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [9]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'company page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'brand assets', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn center', 'url': 'https://huggingface.co/learn'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Endoints API', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'Discord community', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'Chan

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [10]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [11]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-9B
Updated
5 days ago
•
516k
•
533
Qwen/Qwen3.5-35B-A3B
Updated
8 days ago
•
1M
•
1k
Qwen/Qwen3.5-0.8B
Updated
5 days ago
•
265k
•
298
Lightricks/LTX-2.3
Updated
1 day ago
•
35k
•
270
Qwen/Qwen3.5-4B
Updated
5 days ago
•
233k
•
265
Browse 2M+ models
Spaces
Running
on
Zero
Featured
384
Omni Video Factory
🏆
384
text to video, image to video, video extend
Running
on
Zero
MCP
1.12k
Wan2.2 14B Preview
🐌
1.12k
generate a video from an image with a text prompt
Running
on
Zero
125
OBLITERATUS
💥
125
One-click model liberation + chat playground
Running
on
Zero
Featured
1.84k
Qwen Image Multiple Angles 3D Camera
🎥
1.84k
Change 

In [19]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [13]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [14]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-9B\nUpdated\n5 days ago\n•\n516k\n•\n533\nQwen/Qwen3.5-35B-A3B\nUpdated\n8 days ago\n•\n1M\n•\n1k\nQwen/Qwen3.5-0.8B\nUpdated\n5 days ago\n•\n265k\n•\n298\nLightricks/LTX-2.3\nUpdated\n1 day ago\n•\n35k\n•\n270\nQwen/Qwen3.5-4B\nUpdated\n5 days ago\n•\n233k\n•\n265\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n384\nOmni Video Factory\n🏆\n384\ntext to video, image to 

In [15]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [16]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is a leading AI community and collaboration platform dedicated to building the future of machine learning. It provides an open and ethical environment where machine learning engineers, scientists, and end users worldwide can share, explore, and develop models, datasets, and applications.

As the home of machine learning, Hugging Face serves as a central hub with over 2 million models, 500,000+ datasets, and 1 million+ applications across multiple modalities — including text, image, video, audio, and 3D.

---

## What We Offer

- **Collaborative Platform:** Host and collaborate on unlimited public machine learning models, datasets, and applications.
- **Model Library:** Access and contribute to a vast repository of open-source models, including trending ones like Qwen series and FireRed Image Editor.
- **Datasets Collection:** Discover diverse, up-to-date datasets to accelerate research and development.
- **Spaces:** Explore and deploy AI-powered apps quickly using Hugging Face Spaces.
- **Open Source Stack:** Leverage cutting-edge open-source tools to move faster in your ML projects.
- **Enterprise Solutions:** Paid compute and tailor-made enterprise offerings to help teams accelerate ML initiatives efficiently.

---

## Community & Collaboration

Hugging Face fosters a fast-growing, vibrant AI community focused on learning, sharing, and inventing collaboratively. The platform encourages users to build their portfolios by showcasing work openly, engaging with other ML practitioners, and pushing the boundaries of AI innovation together.

---

## Company Culture

- **Open and Ethical AI:** Committed to driving AI development that is responsible, transparent, and inclusive.
- **Collaborative Spirit:** Collaboration is at the heart of what we do, bridging experts and enthusiasts worldwide.
- **Empowerment:** We empower the next generation of machine learning professionals by providing accessible, cutting-edge tools and resources.
- **Innovation:** Constantly evolving with the latest research and technologies to stay at the forefront of AI development.

---

## For Customers and Enterprise Clients

Hugging Face provides scalable enterprise solutions tailored to corporate needs, including:

- Access to premium compute resources.
- Dedicated support for AI model deployment.
- Collaboration tools for teams to streamline machine learning workflows.
- Consultation and customization to integrate Hugging Face tools into business pipelines.

---

## Careers at Hugging Face

Join a global team that is passionate about shaping the future of AI. Whether you are an engineer, researcher, product specialist, or community manager, Hugging Face offers exciting career paths to grow your expertise in a supportive and innovative environment.

- Work on impactful, open-source projects used by millions.
- Collaborate with top-tier ML and AI professionals.
- Engage in a mission-driven culture focused on open, ethical AI.
- Explore roles in engineering, research, business development, and more.

Visit [huggingface.co/careers](https://huggingface.co) to learn more about current openings and apply.

---

## Connect With Us

- Website: https://huggingface.co
- Explore Models & Datasets: https://huggingface.co/models
- Discover AI Apps & Spaces: https://huggingface.co/spaces
- Join the Community: https://huggingface.co/community
- Docs & Resources: https://huggingface.co/docs

---

**Hugging Face — The AI community building the future.**  
Collaborate. Share. Innovate. Together.

---

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [17]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [18]:
stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a leading AI community and collaboration platform dedicated to building the future of machine learning (ML). It serves as the central hub where ML engineers, scientists, and enthusiasts come together to share, explore, and experiment with open-source models, datasets, and applications. Hugging Face empowers the next generation of AI creators to build an open, ethical, and innovative AI future.

---

## What We Offer

- **Vast Model Repository**: Access over **2 million machine learning models** spanning text, image, video, audio, and even 3D modalities.
- **Extensive Datasets**: Browse and utilize from more than **500,000 datasets** to train and improve your models.
- **Innovative Spaces**: Discover and deploy interactive AI apps and demos hosted in **Hugging Face Spaces**, featuring trending tools like text-to-video generators and 3D camera angle changers.
- **Open Source Tools**: Utilize the Hugging Face open source stack designed for faster, more collaborative ML development.
- **Compute & Enterprise Solutions**: Scale your AI projects with Hugging Face’s paid compute services and tailored enterprise offerings.
- **Community Collaboration**: Host unlimited public models, datasets, and applications, collaborate with peers, and expand your professional portfolio on a globally recognized platform.

---

## Community & Culture

- **Open & Ethical AI**: Hugging Face advocates for transparency and ethics in AI development, fostering an inclusive environment where ideas and innovations are shared openly.
- **Collaborative Environment**: Join a fast-growing global community dedicated to pushing the boundaries of machine learning through collaboration and knowledge exchange.
- **Learning & Growth**: Whether you’re an experienced machine learning practitioner or a newcomer, Hugging Face offers numerous ways to build your skills and showcase your work.

---

## Customers & Users

Hugging Face is trusted by individuals, researchers, startups, and enterprises worldwide who work in AI and machine learning. The platform supports a wide variety of use cases including:

- Natural Language Processing (NLP)
- Computer Vision
- Audio Processing
- Video Synthesis and Editing
- Reinforcement Learning and more

Leading organizations use Hugging Face to accelerate AI innovation, leveraging open-source models and tailored enterprise solutions.

---

## Careers & Opportunities

Hugging Face is continuously expanding its talented team of AI researchers, software engineers, and community managers. If you are passionate about shaping the future of AI and enjoy working in a vibrant, community-driven environment, Hugging Face offers exciting career opportunities.

- Work on cutting-edge AI technologies
- Collaborate with inspired and like-minded innovators
- Contribute to impactful open-source projects
- Enjoy a culture that values openness, collaboration, and ethical AI development

---

## Join Us

- **Explore** AI apps and 2M+ models at [huggingface.co](https://huggingface.co)
- **Sign Up** for free to host, share, and collaborate on ML projects
- **Accelerate** your AI journey with Hugging Face’s compute and enterprise offerings

---

## Brand Identity

- Signature colors: Yellow (#FFD21E), Orange (#FF9D00), and Gray (#6B7280)
- Recognized for a friendly and approachable brand symbolizing collaboration and innovation in AI

---

**Hugging Face** — The AI community building the future, today.

In [20]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Welcome to Hugging Face: The AI Community Building the Future

---

## Who Are We?  
Imagine a world where machines don't just compute but *collaborate*, where AI enthusiasts from every corner of the planet can find, build, and share cutting-edge machine learning models and datasets — all in one buzzing digital playground. That’s Hugging Face, your friendly neighborhood AI community, the *open-source Swiss Army knife* for machine learning engineers, scientists, and creators alike.

We’re not just a platform. We’re the social network for your models, the creative studio for your datasets, and the launchpad for your next big AI breakthrough. You bring the curiosity, we bring over **2 million models, 500,000 datasets, and 1 million+ applications**.

---

## What’s in Our Hug? 🤗

### Models Galore  
From tiny bots to colossal brains like **Qwen3.5-35B**, over 2 million models ready to be tested, tweaked, or turbocharged. Trendy, trusted, and sometimes downright theatrical, our models cover text, images, video, audio—even 3D. 

### Datasets & Spaces  
Explore a universe of data to train your models or pit your creativity in cutting-edge AI playgrounds called *Spaces*—where fun meets functionality. Think text-to-video magic or photo angle wizardry—all running on ZERO fuss.

### Collaboration is Our Superpower  
We give you the open-source tools, the compute power, and the freedom to build your AI portfolio and show it off to the world. Because great ideas rarely come alone.

---

## Culture that Sparks Innovation 🔥

- **Open & Ethical AI:** We believe in building technology for good, with transparency, responsibility, and a smile.
- **Fast-Paced & Friendly:** Our community grows every day. We move fast, experiment fearlessly, and share generously.
- **Inclusivity & Learning:** Whether you're a newbie or an AI Nobel laureate-in-the-making, you belong here.
- **Scientists & Engineers, Assemble:** Cutting-edge research meets real-world impact. Our in-house science team dives deep at the frontier of AI.

Plus: Expect yellow (#FFD21E) vibes and a lot of emoji fun because why make AI serious when it can be seriously awesome?

---

## Who Loves Hugging Face?

- **Machine Learning Engineers** looking for the quickest route from prototype to production.
- **Data Scientists & Researchers** hungry for open, reusable datasets and bleeding-edge models.
- **AI Startups & Enterprises** who want scalable compute and enterprise solutions without vendor lock-in.
- **Creators & Hobbyists** eager to explore the latest AI apps or launch an AI Space of their own.
- **Educators & Students** aiming to learn with real-world tools and community support.

---

## Careers: Join the Hugging Revolution!

Ready to turn AI dreams into reality? We're always scouting for passionate engineers, researchers, and makers who want to leave a dent in the universe. Expect a culture where ideas glow brighter than code commits and every lunch conversation might start with “Did you try that new multimodal model?”

Explore current openings and become a part of the family where collaboration beats competition and where your Hugging Face is just a handshake away.

---

## Why Hugging Face?  

Because AI shouldn’t be lonely. We are the *community-driven engine* powering the next generation of applications — from autonomous agents to protein folding, from autonomous driving to creative AI. We provide the **collaborative hub, open-source stack, and scalable tools** to fuel your innovation.

---

## Hugging Face in a Nutshell  
> The AI community where innovation hugs collaboration, datasets meet dreams, and models come with a side of friendly vibes. Come build the future of machine learning with us — because the best AI is the one we build together.

---

**Join us today:**  
[Sign Up & Explore](https://huggingface.co) | Discover the future, one model at a time.  
Embrace the hug. Embrace the future. 🤗

---

### Brand Colors & Vibes  
Sunny Yellow (#FFD21E), Energetic Orange (#FF9D00), and Neutral Cool Gray (#6B7280)—because a brilliant AI brain loves a colorful personality.

---

*Hugging Face: Where Machines Meet Humans, and AI Gets Friendly*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>